In [96]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/customer_churn_dirty (1).csv")

In [97]:
columns = df.select_dtypes(include=['object', 'float64', 'int64']).columns
df = pd.DataFrame(df, columns=columns)

C:\Users\HP\AppData\Local\Temp\ipykernel_23768\3812595828.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns = df.select_dtypes(include=['object', 'float64', 'int64']).columns


In [99]:
churn_df = df.drop(columns=['Payment_Method', 'Customer_ID'])

In [100]:
churn_df['Gender'] = churn_df['Gender'].str.strip().str.lower()

churn_df['Gender'] = churn_df['Gender'].replace(['male', 'm'], 1)
churn_df['Gender'] = churn_df['Gender'].replace(['female', 'f'], 0)
churn_df['Gender'] = churn_df['Gender'].replace(['n/a', 'other'], 2)

churn_df['Gender'] = pd.to_numeric(
    churn_df['Gender'], errors='coerce')

In [101]:
churn_df['Contract_Type'] = churn_df['Contract_Type'].str.strip().str.lower()

churn_df['Contract_Type'] = churn_df['Contract_Type'].replace(
    ['monthly', 'm', 'monthy'], 1)
churn_df['Contract_Type'] = churn_df['Contract_Type'].replace([
                                                              'yearly', 'y'], 2)
churn_df['Contract_Type'] = churn_df['Contract_Type'].replace(
    ['two-year', '2-year', 'two year'], 3)

churn_df['Contract_Type'] = pd.to_numeric(
    churn_df['Contract_Type'], errors='coerce').astype(float)

In [102]:
churn_df['Tenure_Months'] = churn_df['Tenure_Months'].abs()

In [103]:
churn_df['Monthly_Charges'] = pd.to_numeric(
    churn_df['Monthly_Charges'], errors='coerce')
churn_df['Monthly_Charges'] = churn_df['Monthly_Charges'].replace(
    r'[^\d.]', '', regex=True)

In [104]:
churn_df['Age'] = pd.to_numeric(churn_df['Age'], errors='coerce')

median_age = churn_df['Age'].median()
churn_df['Age'] = churn_df['Age'].fillna(median_age)
churn_df['Age'] = churn_df['Age'].astype(int)
churn_df['Age'] = churn_df['Age'].replace(0.0, median_age)

In [105]:
churn_df['Total_Charges'] = pd.to_numeric(
    churn_df['Total_Charges'], errors='coerce')
missing_total_value = churn_df['Tenure_Months'] * churn_df['Monthly_Charges']

churn_df['Total_Charges'] = churn_df['Total_Charges'].fillna(
    missing_total_value)
churn_df['Total_Charges'] = churn_df['Total_Charges'].abs()

In [106]:
churn_df['Internet_Service'] = churn_df['Internet_Service'].str.strip().str.lower()

churn_df['Internet_Service'] = churn_df['Internet_Service'].replace(
    ['fiber', 'fiber optic'], 1)
churn_df['Internet_Service'] = churn_df['Internet_Service'].replace('dsl', 2)
churn_df['Internet_Service'] = churn_df['Internet_Service'].replace(
    ['no service', 'no', 'none', 'n'], 3)

churn_df['Internet_Service'] = pd.to_numeric(
    churn_df['Internet_Service'], errors='coerce').astype(float)

In [107]:
churn_df['Has_Multiple_Lines'] = churn_df['Has_Multiple_Lines'].str.strip().str.lower()

churn_df['Has_Multiple_Lines'] = churn_df['Has_Multiple_Lines'].replace([
                                                                        'yes', 'y'], 1)
churn_df['Has_Multiple_Lines'] = churn_df['Has_Multiple_Lines'].replace([
                                                                        'no', 'n'], 0)


churn_df['Has_Multiple_Lines'] = pd.to_numeric(
    churn_df['Has_Multiple_Lines'], errors='coerce')

In [108]:
churn_df['Streaming_TV'] = churn_df['Streaming_TV'].str.strip().str.lower()

churn_df['Streaming_TV'] = churn_df['Streaming_TV'].replace(['yes', 'y'], 1)
churn_df['Streaming_TV'] = churn_df['Streaming_TV'].replace(['no', 'n'], 0)

churn_df['Streaming_TV'] = pd.to_numeric(
    churn_df['Streaming_TV'], errors='coerce').astype(float)

In [109]:
churn_df['Churn'] = churn_df['Churn'].str.strip().str.lower()

churn_df['Churn'] = churn_df['Churn'].replace(['yes', 'y'], 1)
churn_df['Churn'] = churn_df['Churn'].replace(['no', 'n'], 0)

churn_df['Churn'] = pd.to_numeric(
    churn_df['Churn'], errors='coerce')

In [110]:
churn_df['Reason_for_Churn'] = churn_df.Reason_for_Churn.str.strip().str.lower()

churn_df['Reason_for_Churn'] = churn_df.Reason_for_Churn.fillna(0)
churn_df['Reason_for_Churn'] = churn_df.Reason_for_Churn.replace('price', 1)
churn_df['Reason_for_Churn'] = churn_df.Reason_for_Churn.replace(
    'poor service', 2)
churn_df['Reason_for_Churn'] = churn_df.Reason_for_Churn.replace(
    'competitor', 3)
churn_df['Reason_for_Churn'] = churn_df.Reason_for_Churn.replace('other', 4)

churn_df['Reason_for_Churn'] = pd.to_numeric(
    churn_df['Reason_for_Churn'], errors='coerce')

In [111]:
# churn_df.to_csv('Quantified_dirty_churn.csv', index=False)

In [112]:
quant_df = pd.DataFrame(df, columns=columns)

quant_df['Monthly_Charges'] = pd.to_numeric(
    quant_df['Monthly_Charges'], errors='coerce')
quant_df['Total_Charges'] = pd.to_numeric(
    quant_df['Total_Charges'], errors='coerce')

churn_df = quant_df[quant_df['Monthly_Charges'] <= quant_df['Total_Charges']]

calc_tenure = churn_df['Total_Charges'] / churn_df['Monthly_Charges']
churn_df['Tenure_Months'] = churn_df['Tenure_Months'].fillna(
    calc_tenure.round())

In [113]:
churn_df['Gender'] = churn_df['Gender'].fillna(2)
churn_df['Contract_Type'] = churn_df['Contract_Type'].fillna(1)
churn_df['Internet_Service'] = churn_df['Internet_Service'].fillna(1)
churn_df['Has_Multiple_Lines'] = churn_df['Has_Multiple_Lines'].fillna(1)
churn_df['Streaming_TV'] = churn_df['Streaming_TV'].fillna(1)

churn_df = churn_df.dropna(subset=['Churn'])

In [114]:
X_feature = churn_df.drop(columns='Churn')
y_churn = churn_df['Churn']

In [115]:
# X_feature.to_csv('X_feature.csv', index=False)
# y_churn.to_csv('y_churn.csv', index=False)